# 📡 TP Télécom 3GPP — Phase 7 : Multi-Agents + RAFT

**Objectif** : Déployer plusieurs agents spécialisés qui collaborent,
chacun utilisant le modèle RAFT entraîné en Phase 5.

**Architecture Multi-Agents :**
```
Question
    ↓
Agent Orchestrateur (décide qui répond)
    ↓
┌─────────────────────────────────────┐
│  Agent Retriever  → cherche docs    │
│  Agent Analyste   → analyse question│
│  Agent Générateur → génère réponse  │
│  Agent Validateur → vérifie réponse │
└─────────────────────────────────────┘
    ↓
Réponse finale consolidée
```

```
Phase 1 ✅ → Phase 2 ✅ → Phase 3 ✅ → Phase 4 ✅ → Phase 5 ✅ → Phase 6 ✅ → [Phase 7]
```

## 1. Installation des dépendances

In [ ]:
!pip install -q transformers torch peft
!pip install -q faiss-cpu sentence-transformers rank-bm25
!pip install -q evaluate rouge_score
!pip install -q pandas matplotlib
print('✅ Installation terminée')

## 2. Imports & Configuration

In [ ]:
import json, time
import numpy as np
import pandas as pd
import faiss
import matplotlib.pyplot as plt
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from peft import PeftModel
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import warnings
warnings.filterwarnings('ignore')

SAVE_DIR = r'C:\Users\HP\Documents\TP-LLM-3GPP-Pipeline'
print('✅ Imports effectués')

## 3. Chargement du Modèle RAFT (Phase 5)

In [ ]:
# Config
with open(f'{SAVE_DIR}\\pipeline_config.json') as f:
    config = json.load(f)
MODEL_ID = config['best_model_id']
print(f'✅ Modèle base : {config["best_model_name"]}')

# Chargement modèle RAFT
print('🔄 Chargement du modèle RAFT...')
tokenizer  = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32)
raft_model = PeftModel.from_pretrained(base_model, f'{SAVE_DIR}\\Phase5\\model_raft')
raft_pipe  = pipeline('text-generation', model=raft_model,
                       tokenizer=tokenizer, pad_token_id=50256)
print('✅ Modèle RAFT chargé')

# Dataset
TRAIN_PATH = r'C:\Users\HP\Downloads\TeleQnA_training.txt'
TEST_PATH  = r'C:\Users\HP\Downloads\TeleQnA_testing.txt'
with open(TRAIN_PATH, 'r', encoding='utf-8') as f:
    train_data = json.load(f)
with open(TEST_PATH, 'r', encoding='utf-8') as f:
    test_data = json.load(f)
print(f'✅ Dataset chargé')

## 4. Base de Connaissances

In [ ]:
# Embedding
print('🔄 Chargement embedding...')
embed_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Corpus
corpus = []
for key, item in train_data.items():
    answer_idx  = item.get('answer', 1)
    answer_text = item.get(f'option {answer_idx}', str(answer_idx))
    corpus.append({
        'text'    : f"Question: {item['question']} Answer: {answer_text}. {item.get('explanation', '')}",
        'category': item.get('category', '3GPP'),
        'answer'  : str(answer_text)
    })

# FAISS
texts      = [doc['text'] for doc in corpus]
embeddings = embed_model.encode(texts, batch_size=64, show_progress_bar=True)
embeddings = np.array(embeddings).astype('float32')
faiss.normalize_L2(embeddings)
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

# BM25
tokenized = [doc['text'].lower().split() for doc in corpus]
bm25      = BM25Okapi(tokenized)

print(f'✅ Base de connaissances : {len(corpus)} documents')

## 5. Définition des 4 Agents Spécialisés

In [ ]:
# ============================================================
# AGENT 1 : Retriever — Cherche les documents pertinents
# ============================================================
class AgentRetriever:
    def __init__(self, name='Retriever'):
        self.name = name

    def run(self, question, top_k=3):
        # Dense
        q_emb = embed_model.encode([question]).astype('float32')
        faiss.normalize_L2(q_emb)
        scores, indices = index.search(q_emb, top_k)
        docs = [{'text': corpus[i]['text'],
                 'category': corpus[i]['category'],
                 'score': float(s)}
                for s, i in zip(scores[0], indices[0])]
        context = ' | '.join([d['text'][:150] for d in docs])
        return {'docs': docs, 'context': context,
                'avg_score': round(np.mean([d['score'] for d in docs]), 4)}

# ============================================================
# AGENT 2 : Analyste — Analyse la question
# ============================================================
class AgentAnalyste:
    def __init__(self, name='Analyste'):
        self.name = name

    def run(self, question):
        # Détection catégorie
        keywords = {
            'RRC/Radio'    : ['rrc', 'radio', 'phy', 'mac', 'pdcp', 'nr', 'lte'],
            'Core Network' : ['amf', 'smf', 'upf', 'pcf', 'core', '5gc'],
            'Security'     : ['security', 'authentication', 'encryption', 'aka'],
            'Services'     : ['service', 'slice', 'qos', 'session', 'pdu'],
            'General 3GPP' : ['3gpp', 'release', 'standard']
        }
        q_lower = question.lower()
        scores  = {c: sum(1 for w in ws if w in q_lower) for c, ws in keywords.items()}
        category = max(scores, key=scores.get)

        # Type de question
        q_type = 'definition' if any(w in q_lower for w in ['what is', 'define', 'explain']) \
            else 'procedure' if any(w in q_lower for w in ['how', 'when', 'process']) \
            else 'comparison' if any(w in q_lower for w in ['difference', 'compare', 'versus']) \
            else 'factual'

        return {'category': category, 'question_type': q_type,
                'complexity': 'high' if len(question.split()) > 20 else 'low'}

# ============================================================
# AGENT 3 : Générateur — Génère la réponse avec RAFT
# ============================================================
class AgentGenerateur:
    def __init__(self, name='Générateur'):
        self.name = name

    def run(self, question, context, max_new_tokens=100):
        prompt = f'Context: {context[:300]} Question: {question} Answer:'
        result = raft_pipe(
            prompt, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=50256, truncation=True
        )
        answer = result[0]['generated_text'].replace(prompt, '').strip()
        return {'answer': answer, 'prompt_length': len(prompt)}

# ============================================================
# AGENT 4 : Validateur — Valide la qualité de la réponse
# ============================================================
class AgentValidateur:
    def __init__(self, name='Validateur'):
        self.name = name

    def run(self, question, answer, docs):
        # Vérifications
        is_empty     = len(answer.strip()) < 5
        is_repetitive = answer.count(answer[:20]) > 2 if len(answer) > 20 else False
        has_3gpp_terms = any(term in answer.lower() for term in
                             ['3gpp', 'lte', '5g', 'nr', 'ue', 'gnb', 'amf', 'rrc'])
        retrieval_quality = np.mean([d['score'] for d in docs]) if docs else 0

        quality_score = 0
        if not is_empty      : quality_score += 0.3
        if not is_repetitive : quality_score += 0.3
        if has_3gpp_terms    : quality_score += 0.2
        quality_score += min(0.2, retrieval_quality * 0.2)

        return {
            'quality_score'     : round(quality_score, 3),
            'is_valid'          : quality_score > 0.4,
            'has_3gpp_terms'    : has_3gpp_terms,
            'retrieval_quality' : round(float(retrieval_quality), 4)
        }

# Instanciation des agents
agent_retriever  = AgentRetriever()
agent_analyste   = AgentAnalyste()
agent_generateur = AgentGenerateur()
agent_validateur = AgentValidateur()

print('✅ 4 agents spécialisés créés :')
print('   🔍 AgentRetriever  — Recherche RAG')
print('   🧠 AgentAnalyste   — Analyse question')
print('   ✍️  AgentGenerateur — Génère réponse RAFT')
print('   ✅ AgentValidateur  — Valide la réponse')

## 6. Agent Orchestrateur — Coordination des Agents

In [ ]:
def orchestrateur(question, verbose=True):
    """
    Orchestrateur Multi-Agents :
    Coordonne les 4 agents pour produire la meilleure réponse.
    """
    t0    = time.time()
    trace = []

    if verbose:
        print(f'\n🎯 ORCHESTRATEUR — Traitement de la question')
        print('='*60)

    # ── Étape 1 : Analyse ──
    analysis = agent_analyste.run(question)
    trace.append(f'[Analyste] Catégorie: {analysis["category"]} | Type: {analysis["question_type"]}')
    if verbose:
        print(f'  🧠 Analyste    → Catégorie: {analysis["category"]} | Type: {analysis["question_type"]}')

    # ── Étape 2 : Retrieval ──
    retrieval = agent_retriever.run(question, top_k=3)
    trace.append(f'[Retriever] {len(retrieval["docs"])} docs | Score moy: {retrieval["avg_score"]}')
    if verbose:
        print(f'  🔍 Retriever   → {len(retrieval["docs"])} docs trouvés | Score: {retrieval["avg_score"]}')

    # ── Étape 3 : Génération RAFT ──
    generation = agent_generateur.run(question, retrieval['context'])
    trace.append(f'[Générateur] Réponse générée ({len(generation["answer"])} chars)')
    if verbose:
        print(f'  ✍️  Générateur  → Réponse: {generation["answer"][:80]}...')

    # ── Étape 4 : Validation ──
    validation = agent_validateur.run(question, generation['answer'], retrieval['docs'])
    trace.append(f'[Validateur] Score: {validation["quality_score"]} | Valide: {validation["is_valid"]}')
    if verbose:
        print(f'  ✅ Validateur  → Score qualité: {validation["quality_score"]} | Valide: {validation["is_valid"]}')

    elapsed = time.time() - t0
    return {
        'answer'            : generation['answer'],
        'category'          : analysis['category'],
        'question_type'     : analysis['question_type'],
        'retrieval_score'   : retrieval['avg_score'],
        'quality_score'     : validation['quality_score'],
        'is_valid'          : validation['is_valid'],
        'has_3gpp_terms'    : validation['has_3gpp_terms'],
        'trace'             : trace,
        'time'              : round(elapsed, 2)
    }

print('✅ Orchestrateur défini')

## 7. Test sur une Question

In [ ]:
test_items = list(test_data.values())
test_q     = test_items[0]['question']

print(f'❓ Question : {test_q}')
result = orchestrateur(test_q, verbose=True)

print('\n' + '='*60)
print('📋 Trace complète :')
for step in result['trace']:
    print(f'   {step}')
print(f'\n💬 Réponse finale : {result["answer"][:300]}')
print(f'⏱️  Temps total   : {result["time"]}s')

## 8. Évaluation sur 10 Questions

In [ ]:
results = []
print('🚀 Évaluation Multi-Agents + RAFT sur 10 questions...')

for i, item in enumerate(test_items[:10]):
    question    = item['question']
    answer_idx  = item.get('answer', item.get('option 1', ''))
    answer_text = item.get(f'option {answer_idx}', str(answer_idx)) if isinstance(answer_idx, int) else str(answer_idx)

    print(f'\n🔍 Q{i+1} : {question[:60]}...')
    result = orchestrateur(question, verbose=False)

    results.append({
        'id'             : i + 1,
        'question'       : question,
        'reference'      : str(answer_text),
        'answer'         : result['answer'],
        'category'       : result['category'],
        'question_type'  : result['question_type'],
        'retrieval_score': result['retrieval_score'],
        'quality_score'  : result['quality_score'],
        'is_valid'       : result['is_valid'],
        'has_3gpp_terms' : result['has_3gpp_terms'],
        'time'           : result['time']
    })
    print(f'   ✓ Qualité:{result["quality_score"]} | Valide:{result["is_valid"]} | {result["time"]}s')

df_results = pd.DataFrame(results)
df_results.to_csv(f'{SAVE_DIR}\\Phase7\\phase7_results.csv', index=False)
print(f'\n✅ {len(df_results)} résultats sauvegardés')
print(f'📊 Réponses valides : {df_results["is_valid"].sum()}/10')
print(f'📊 Score qualité moyen : {df_results["quality_score"].mean():.3f}')

## 9. Évaluation ROUGE — Comparaison Complète

In [ ]:
from evaluate import load as load_metric
rouge  = load_metric('rouge')
refs   = df_results['reference'].tolist()
preds  = df_results['answer'].tolist()
scores = rouge.compute(predictions=preds, references=refs)

# Chargement résultats phases précédentes
phase5_eval = pd.read_csv(f'{SAVE_DIR}\\Phase5\\phase5_evaluation.csv')
phase6_eval = pd.read_csv(f'{SAVE_DIR}\\Phase6\\phase6_evaluation.csv')

# Tableau comparatif complet
all_results = []
for _, row in phase5_eval.iterrows():
    all_results.append(row.to_dict())
all_results.append({
    'Approche'      : 'Agent RAG (P6)',
    'ROUGE-1'       : phase6_eval['ROUGE-1'].values[0],
    'ROUGE-2'       : phase6_eval['ROUGE-2'].values[0],
    'ROUGE-L'       : phase6_eval['ROUGE-L'].values[0],
    'Temps moy. (s)': phase6_eval['Temps moy. (s)'].values[0]
})
all_results.append({
    'Approche'      : 'Multi-Agents+RAFT',
    'ROUGE-1'       : round(scores['rouge1'], 4),
    'ROUGE-2'       : round(scores['rouge2'], 4),
    'ROUGE-L'       : round(scores['rougeL'], 4),
    'Temps moy. (s)': round(df_results['time'].mean(), 2)
})

df_final = pd.DataFrame(all_results)
print('📊 COMPARAISON FINALE — TOUTES LES APPROCHES :')
print(df_final.to_string(index=False))
df_final.to_csv(f'{SAVE_DIR}\\Phase7\\phase7_comparaison_finale.csv', index=False)
print('\n💾 Sauvegardé → Phase7/phase7_comparaison_finale.csv')

## 10. Visualisation Finale Complète

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Comparaison Finale — Toutes les Approches\nDataset 3GPP Télécom',
             fontsize=14, fontweight='bold')

colors  = ['#FF7043','#26A69A','#1565C0','#7B1FA2','#F57F17','#2E7D32']
n       = len(df_final)
palette = colors[:n]

# --- ROUGE-L ---
ax1 = axes[0]
bars = ax1.barh(df_final['Approche'], df_final['ROUGE-L'], color=palette, alpha=0.85)
for b in bars:
    ax1.text(b.get_width()+0.001, b.get_y()+b.get_height()/2,
             f'{b.get_width():.4f}', va='center', fontsize=9, fontweight='bold')
ax1.set_title('ROUGE-L par Approche', fontweight='bold')
ax1.set_xlabel('Score ROUGE-L')
ax1.grid(axis='x', alpha=0.3)

# --- Temps ---
ax2 = axes[1]
bars = ax2.barh(df_final['Approche'], df_final['Temps moy. (s)'], color=palette, alpha=0.85)
for b in bars:
    ax2.text(b.get_width()+0.1, b.get_y()+b.get_height()/2,
             f'{b.get_width():.1f}s', va='center', fontsize=9, fontweight='bold')
ax2.set_title("Temps moyen d'inférence", fontweight='bold')
ax2.set_xlabel('Secondes')
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}\\Phase7\\phase7_comparaison_finale.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Graphique → Phase7/phase7_comparaison_finale.png')

---
## 🎉 Pipeline Complet Terminé !

### Récapitulatif des 7 Phases

| Phase | Approche | Technique |
|-------|----------|----------|
| 1 | Comparaison LLMs | DistilGPT2, GPT2-Small, GPT2-Medium |
| 2 | LLM + RAG | FAISS Dense Retrieval |
| 3 | RAG Avancé | BM25 + FAISS + Reranking |
| 4 | Fine-Tuning | QLoRA sur données 3GPP |
| 5 | RAFT | RAG + Fine-Tuning combinés |
| 6 | Agent RAG | Agent avec 4 outils |
| 7 | Multi-Agents+RAFT | 4 agents spécialisés + RAFT |

### Fichiers produits
- `Phase7/phase7_results.csv`
- `Phase7/phase7_comparaison_finale.csv`
- `Phase7/phase7_comparaison_finale.png`

**➡️ Prochaine étape : Notebook de Synthèse et Rapport Final**